## Example 1 — the import, the version and the kernel sources the package ships

In [ ]:
import importlib.metadata, os
import flashinfer
from flashinfer.jit import env as jit_env

print("flashinfer", flashinfer.__version__)
assert flashinfer.__version__ == importlib.metadata.version("flashinfer-python")
for folder in (jit_env.FLASHINFER_CSRC_DIR, jit_env.FLASHINFER_INCLUDE_DIR):
    assert os.path.isdir(folder) and os.listdir(folder), f"the kernel sources are missing: {folder}"
assert os.path.isfile(os.path.join(jit_env.FLASHINFER_CSRC_DIR, "norm.cu")), "csrc/norm.cu is missing"
for name in ("rmsnorm", "single_decode_with_kv_cache", "BatchDecodeWithPagedKVCacheWrapper",
             "BatchPrefillWithPagedKVCacheWrapper", "top_k_top_p_sampling_from_probs", "apply_rope"):
    assert callable(getattr(flashinfer, name)), name
packs = [provider for provider in jit_env.FLASHINFER_AOT_PROVIDERS]
print(f"OK -- kernel sources under {jit_env.FLASHINFER_DATA}; {len(packs)} pre-built kernel pack(s) found")

## Example 2 — RMSNorm on the GPU, checked against torch

In [ ]:
import torch, flashinfer
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
torch.manual_seed(0)
x = torch.randn(8, 256, device="cuda", dtype=torch.float16)
weight = torch.randn(256, device="cuda", dtype=torch.float16)
out = flashinfer.rmsnorm(x, weight, eps=1e-6)
reference = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + 1e-6) * weight.float()
assert out.shape == x.shape and out.dtype == x.dtype, (out.shape, out.dtype)
torch.testing.assert_close(out.float(), reference, rtol=1e-2, atol=1e-2)
residual = torch.randn_like(x)
expected_residual = residual.float() + x.float()
hidden = x.clone()
flashinfer.fused_add_rmsnorm(hidden, residual, weight, eps=1e-6)  # in place: residual += x; x = rmsnorm(residual)
torch.testing.assert_close(residual.float(), expected_residual, rtol=1e-2, atol=1e-2)
print(f"OK -- rmsnorm matches torch (max difference {(out.float() - reference).abs().max().item():.1e}); the fused add ran in place")

## Example 3 — sampling: top-k / top-p picks stay inside the allowed set

In [ ]:
import torch, flashinfer
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
generator = torch.Generator(device="cuda").manual_seed(1234)
probs = torch.softmax(torch.randn(16, 5000, device="cuda"), dim=-1)
picked = flashinfer.sampling.top_k_top_p_sampling_from_probs(probs, 20, 0.9, generator=generator)
assert picked.shape == (16,), picked.shape
allowed = probs.topk(20, dim=-1).indices
assert all(int(picked[row]) in set(allowed[row].tolist()) for row in range(16)), "a pick fell outside the top 20"
peaked = torch.zeros(4, 100, device="cuda")
peaked[torch.arange(4), torch.tensor([3, 14, 15, 92])] = 1.0  # all the mass on one token per row
certain = flashinfer.sampling.sampling_from_probs(peaked, generator=generator)
assert certain.tolist() == [3, 14, 15, 92], certain.tolist()
renormalised = flashinfer.sampling.top_k_renorm_probs(probs, 10)
assert ((renormalised > 0).sum(-1) == 10).all() and torch.allclose(renormalised.sum(-1), torch.ones(16, device="cuda"), atol=1e-4)
print("OK -- 16 top-k/top-p picks are inside their top 20; a one-token distribution always returns that token")

## Example 4 — attention over a key/value cache, checked against torch

In [ ]:
import torch, flashinfer
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
torch.manual_seed(0)
heads, head_dim, cached_tokens = 4, 64, 37
q = torch.randn(heads, head_dim, device="cuda", dtype=torch.float16)
k = torch.randn(cached_tokens, heads, head_dim, device="cuda", dtype=torch.float16)
v = torch.randn(cached_tokens, heads, head_dim, device="cuda", dtype=torch.float16)
out = flashinfer.single_decode_with_kv_cache(q, k, v)  # one decode step: every head attends to the whole cache
weights = torch.softmax(torch.einsum("hd,lhd->hl", q.float(), k.float()) / head_dim**0.5, dim=-1)
reference = torch.einsum("hl,lhd->hd", weights, v.float())
assert out.shape == (heads, head_dim) and out.dtype == torch.float16, (out.shape, out.dtype)
torch.testing.assert_close(out.float(), reference, rtol=2e-2, atol=2e-2)
print(f"OK -- decode attention over {cached_tokens} cached tokens matches torch (max difference {(out.float() - reference).abs().max().item():.1e})")